In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    f"DATABASE={os.getenv('DB_NAME')};"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)


# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [91]:
query_com = """
SELECT DISTINCT Pedido,
    c.Cliente,
    c.Nombres + ' ' + c.Apellidos AS Nombre,
    c.Email AS Email,
    c.Celular AS Celular,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2026-02-27'
    AND v.Fecha <= '2026-02-28'
    AND Ubicacion = 'Social Selling'
    --and Descripcion LIKE '%KIT%'
    AND v.Venta_Neta > 0
GROUP BY  Pedido,
    c.Cliente,
    c.Nombres + ' ' + c.Apellidos,
    c.Email,
    c.Celular,
    v.Fecha,
    v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_com, engine)

df_ventas.head()

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,4354278515,C63293020,ISABEL MEJIA,isabelmejia2007@hotmail.com,3008164765,2026-02-27,Chat Center,1312184.87
1,4354278692,C811044543,GRAZZ SAS / NO APLICA,gerenciagrazz@gmail.com,3207251689,2026-02-27,Chat Center,420756.30
2,4354278744,C98451177,LUIS FERNANDO ARROYAVE,fernandoarroyaveescobar@hotmail.com,3136601190,2026-02-27,Chat Center,173109.24
3,4354278758,C52867285,JOHANNA PEREZ RAMIREZ,johannaperezr26@hotmail.com,3124675689,2026-02-27,Chat Center,89915.96
4,4354278769,C1019080235,ANGELA MARIA BENAVIDES LUNA,maribelu-2128@hotmail.com,3125856868,2026-02-27,Chat Center,189075.63


In [92]:
# #consulta segmentada 
# query_data = """
# SELECT DISTINCT c.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2025-10-12'
#     AND v.Fecha <= '2025-10-15'
#     AND Canal IN ('Chat Center','CHATCENTER WEB')
#     AND v.Venta_Neta > 0
#     AND Descripcion LIKE '%KIT%'
# GROUP BY  c.Cliente,
#        c.Nombres + ' ' + c.Apellidos,
#        c.Email,
#        c.Celular,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)

# df_ventas.head()

In [93]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT Pedido,
#     v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta,
# 	Canal,
#     SUM(Venta_Neta) AS Venta 
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE (Codigo_Descuento = '0001 Cumpleaños Cliente 25%' or Codigo_Descuento = 'CUMP_FEB2026')
# 	AND Fecha BETWEEN '2026-01-26' AND '2026-01-28'
#     AND Canal IN ('Chat Center','CHATCENTER WEB')
# GROUP BY  Pedido,
#     v.Cliente,
#     cc.Nombres + ' ' +  cc.Apellidos,
#     cc.Email,
#     cc.Celular,
#     v.Fecha,
#     v.Canal
# """


# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [94]:
# Leer archivo
df_social = pd.read_excel("Atribucion.xlsx")

df_social['telefono'] = df_social['telefono'].astype(str).str[2:]

In [95]:
# Normalizar df_ventas
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por celular
df_atribucion = pd.merge(df_ventas, df_social, left_on='Celular', right_on='telefono' , how='inner')

# Sumar la venta total atribuida
total_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {total_venta:,.2f}")
# Resultado final
df_atribucion.head()

Total de Ventas: 2
Total de venta atribuida: 1,535,228.10


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,telefono
0,96695,C83003739-6,NANCY GOMEZ,milenasalests@hotmail.com,3176448204,2026-02-27,CHATCENTER WEB,897440.4,3176448204
1,96741,C52414771,MARIA ANGELICA PERSICH BARRERA,lapersa@hotmail.com,3135322407,2026-02-28,CHATCENTER WEB,637787.7,3135322407


In [96]:
df_atribucion.to_excel('Ventas.xlsx', index=False)